In [ ]:
import os
import sys
from pathlib import Path

import gc

import os; os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from trl import SFTConfig, SFTTrainer

In [ ]:
# Ensure notebook runs from the project root (one level up from notebooks/)
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Project Root:", Path.cwd())

# --- Domain & Model Configuration ---
DOMAIN = "metformin"

# Base weights source: The Stage 1 domain-adapted model merged previously
stage1_merged_dir = Path(f"./experiments/{DOMAIN}/stage1_domain/merged_model")

# Stage 2 targets: Where instruction tuning artifacts are saved
stage2_dir = Path(f"./experiments/{DOMAIN}/stage2_instruct")
stage2_adapter = stage2_dir / "adapter"
stage2_merged = stage2_dir / "merged_model"
stage2_checkpoints = stage2_dir / "checkpoints"

stage2_dir.mkdir(parents=True, exist_ok=True)
stage2_checkpoints.mkdir(parents=True, exist_ok=True)

assert stage1_merged_dir.exists(), f"Error: Stage 1 merged model not found at {stage1_merged_dir}"

print(f"Base Model Path:    {stage1_merged_dir.resolve()}")
print(f"Stage 2 Output Dir: {stage2_dir.resolve()}")

In [ ]:
# Load tokenizer directly from the Stage 1 merged model
tokenizer = AutoTokenizer.from_pretrained(str(stage1_merged_dir))

# TinyLlama requires pad_token to be set explicitly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

In [ ]:
print("Testing Stage 1 base model before instruction fine-tuning...")

stage1_model = AutoModelForCausalLM.from_pretrained(
    str(stage1_merged_dir),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda",
)
stage1_model.eval()

test_prompt = "### Instruction:\nExplain the mechanism of action of Metformin.\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = stage1_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

gen_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
print("\n--- Baseline Output (Raw Completion Engine) ---")
print(tokenizer.decode(gen_tokens, skip_special_tokens=True).strip())

# Free GPU memory immediately before training setup
del stage1_model, inputs, outputs, gen_tokens
gc.collect()
torch.cuda.empty_cache()
print("\nVRAM freed successfully.")

In [ ]:
# Locate instruction dataset
dataset_path = Path("data/pharma_instruction_data.csv")
if not dataset_path.exists():
    dataset_path = Path("pharma_instruction_data.csv")

assert dataset_path.exists(), f"Could not find dataset at {dataset_path.resolve()}"
print(f"Loading dataset from: {dataset_path.resolve()}")

raw_dataset = load_dataset("csv", data_files=str(dataset_path), split="train")

# 90/10 split for monitoring loss & validation curves
dataset_split = raw_dataset.train_test_split(test_size=0.1, seed=42)
raw_train = dataset_split["train"]
raw_eval = dataset_split["test"]

print(f"Train samples: {len(raw_train)} | Validation samples: {len(raw_eval)}")

In [ ]:
def format_to_prompt_completion(example):
    instruction = example.get("instruction", "").strip()
    context = example.get("input", "")
    output = example.get("output", "").strip()

    # Format user prompt
    if context and context.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{context.strip()}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    # Crucial: explicitly terminate the completion with tokenizer.eos_token (</s>)
    completion = f"{output}{tokenizer.eos_token}"

    return {
        "prompt": prompt,
        "completion": completion,
    }

train_dataset = raw_train.map(format_to_prompt_completion, remove_columns=raw_train.column_names)
eval_dataset = raw_eval.map(format_to_prompt_completion, remove_columns=raw_eval.column_names)

print("=== Formatted Sample 0 ===")
print("PROMPT:\n", train_dataset[0]["prompt"])
print("COMPLETION:\n", train_dataset[0]["completion"])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading Stage 1 base model from {stage1_merged_dir}...")
base_model = AutoModelForCausalLM.from_pretrained(
    str(stage1_merged_dir),
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

instruction_model = get_peft_model(base_model, lora_config)

# Cast all trainable adapter weights to float32 to prevent AMP unscale errors
for param in instruction_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

instruction_model.print_trainable_parameters()

In [ ]:
# Automatically detect if your GPU supports native BF16 (RTX 30xx/40xx)
use_bf16 = torch.cuda.is_bf16_supported()

sft_config = SFTConfig(
    output_dir=str(stage2_checkpoints),
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    fp16=not use_bf16,              # Use FP16 only if BF16 is not supported
    bf16=use_bf16,                  # Native BF16 avoids GradScaler entirely
    logging_strategy="steps",
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
    max_length=512,
    completion_only_loss=False,     # Bypasses TRL's negative gather index crash
)

trainer = SFTTrainer(
    model=instruction_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting instruction fine-tuning...")
trainer.train()

# Save final best adapter and tokenizer
print(f"Saving final adapter to: {stage2_adapter}...")
trainer.save_model(str(stage2_adapter))
tokenizer.save_pretrained(str(stage2_adapter))

In [ ]:
# 1. Clean training graph from GPU
del instruction_model, base_model, trainer
gc.collect()
torch.cuda.empty_cache()

print("1. Loading clean Stage 1 base model in full FP16...")
clean_stage1 = AutoModelForCausalLM.from_pretrained(
    str(stage1_merged_dir),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda",
)

print(f"2. Loading Stage 2 adapter from: {stage2_adapter}...")
lora_model = PeftModel.from_pretrained(clean_stage1, str(stage2_adapter))

print("3. Merging weights...")
final_merged_model = lora_model.merge_and_unload()

print(f"4. Saving merged standalone model to: {stage2_merged}...")
final_merged_model.save_pretrained(str(stage2_merged), safe_serialization=True)
tokenizer.save_pretrained(str(stage2_merged))

print("[SUCCESS] Stage 2 model merged and saved ready for export!")

In [ ]:
test_questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "What are the most common adverse side effects associated with Metformin therapy?"
]

def generate_response(model, prompt_str, max_tokens=100):
    inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

print("=" * 70)
print("INFERENCE EVALUATION: STAGE 2 INSTRUCTION MODEL")
print("=" * 70)

for q in test_questions:
    instruct_prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    response = generate_response(final_merged_model, instruct_prompt)
    print(f"\n[Question]: {q}")
    print(f"[Response]:\n{response}\n" + "-" * 50)

In [ ]:
test_prompt = "### Instruction:\nWhat are the primary clinical indications for Metformin?\n\n### Response:\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = final_merged_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
print("\n--- Test Generation ---")
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())